# 90 - Configura??es de TF-IDF, clusteriza??o e avalia??o

Este notebook re?ne e documenta os par?metros para planejar experimentos de agrupamento das descri??es de itens fiscais. A unidade de an?lise ? uma `descricao_normalizada` ?nica; `frequencia` ? usada para descrever a import?ncia dos grupos, mas n?o pondera o ajuste.

O objetivo ? revisar os valores da grade e seus efeitos antes de executar o experimento completo. Para treinar os modelos, gerar revis?es de clusters e avaliar categorias com r?tulos humanos, use `70_matriz_tfidf_kmeans_e_validacao.ipynb`, que implementa esse fluxo e registra seus artefatos por execu??o.


## 1. Par?metros do experimento

As representa??es cobrem palavras (unigramas e unigramas com bigramas) e fragmentos de caracteres. `min_df` descarta termos raros. `max_features` limita o tamanho do vocabul?rio. `k` determina a quantidade de clusters; as sementes permitem observar a varia??o da inicializa??o.

A reserva de valida??o ? feita por descri??o ?nica antes de ajustar o TF-IDF. Ela mede desempenho em descri??es n?o vistas da mesma base, n?o generaliza??o temporal. A classifica??o final depende de r?tulos humanos para os clusters e para os itens reservados.


In [1]:
from itertools import product

import pandas as pd
from IPython.display import display

from nofis_classifier.datasets import (
    DEFAULT_PERIOD_LABEL,
    FILTERED_MARKET_DATASET_NAME,
    load_processed_items,
    prepare_modeling_items,
    processed_dataset_path,
)
from nofis_classifier.paths import find_project_root

# Origem dos dados
DATASET_NAME = FILTERED_MARKET_DATASET_NAME
PERIOD_LABEL = DEFAULT_PERIOD_LABEL
VALIDATION_COUNT = 300
SPLIT_SEED = 42
MAX_DESCRIPTIONS = None  # None usa a base inteira; um inteiro limita para explora??o.

# Representa??es TF-IDF: (analisador, intervalo de n-gramas, limites de features)
REPRESENTATIONS = [
    ("word", (1, 1), [10_000]),
    ("word", (1, 2), [10_000]),
    ("char_wb", (3, 5), [30_000]),
]
MIN_DF_VALUES = [2, 5]

# K-Means
K_VALUES = [10, 15, 20, 25, 30, 40, 60, 80]
SEEDS = [42, 123, 2026]
N_INIT = 10
MAX_ITER = 300

# Avalia??o interna: mesma amostra de treino para todas as combina??es.
METRIC_SAMPLE_SIZE = 1_000
DENSE_LIMIT_MB = 512
TOP_COMBINATIONS = 3

PROJECT_ROOT = find_project_root()
DATASET_PATH = processed_dataset_path(DATASET_NAME, PERIOD_LABEL, root=PROJECT_ROOT)
DATASET_PATH


WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais-filtros-combinados/items-notas-fiscais-filtros-combinados-202501-202504.parquet')

## 2. Conferir a base e a viabilidade dos par?metros

A confer?ncia abaixo carrega somente os dados processados e calcula o tamanho previsto da grade. Ela n?o ajusta vetorizadores nem modelos. Ajuste `K_VALUES` se algum `k` n?o for menor que a quantidade de descri??es de treino.


In [2]:
items = prepare_modeling_items(load_processed_items(DATASET_PATH))
if MAX_DESCRIPTIONS is not None:
    items = items.sample(n=min(MAX_DESCRIPTIONS, len(items)), random_state=SPLIT_SEED).reset_index(drop=True)

if not 0 <= VALIDATION_COUNT < len(items):
    raise ValueError("VALIDATION_COUNT deve ser n?o negativo e menor que a quantidade de descri??es.")
training_count = len(items) - VALIDATION_COUNT
invalid_k = [k for k in K_VALUES if not 2 <= k < training_count]
if invalid_k:
    raise ValueError(f"Valores de k incompat?veis com {training_count} descri??es de treino: {invalid_k}")

planned = []
for analyzer, ngram_range, feature_limits in REPRESENTATIONS:
    for min_df, max_features, k, seed in product(
        MIN_DF_VALUES, feature_limits, K_VALUES, SEEDS
    ):
        planned.append({
            "analyzer": analyzer,
            "ngram_range": str(ngram_range),
            "min_df": min_df,
            "max_features": max_features,
            "k": k,
            "seed": seed,
            "n_init": N_INIT,
        })
experiment_matrix = pd.DataFrame(planned)

display(pd.DataFrame({
    "descricoes_unicas": [len(items)],
    "descricoes_treino": [training_count],
    "descricoes_reservadas": [VALIDATION_COUNT],
    "ajustes_planejados": [len(experiment_matrix)],
}))
display(experiment_matrix.head(12))


,descricoes_unicas,descricoes_treino,descricoes_reservadas,ajustes_planejados
0,27800,27500,300,144


,analyzer,ngram_range,min_df,max_features,k,seed,n_init
0,word,"(1, 1)",2,10000,10,42,10
1,word,"(1, 1)",2,10000,10,123,10
2,word,"(1, 1)",2,10000,10,2026,10
3,word,"(1, 1)",2,10000,15,42,10
4,word,"(1, 1)",2,10000,15,123,10
5,word,"(1, 1)",2,10000,15,2026,10
6,word,"(1, 1)",2,10000,20,42,10
7,word,"(1, 1)",2,10000,20,123,10
8,word,"(1, 1)",2,10000,20,2026,10
9,word,"(1, 1)",2,10000,25,42,10


## 3. Crit?rios para interpretar a avalia??o

- **Silhouette cosseno:** maior indica melhor separa??o m?dia na amostra; n?o ? acur?cia.
- **Calinski-Harabasz:** maior indica maior separa??o relativa ? dispers?o.
- **Davies-Bouldin:** menor indica menor sobreposi??o relativa.
- **In?rcia:** compare o cotovelo somente dentro da mesma representa??o e base.
- **Varia??o entre sementes:** o desvio das m?tricas mostra sensibilidade ? inicializa??o.
- **Valida??o com r?tulos humanos:** reporte cobertura, acur?cia incluindo absten??es (`REVISAR`), acur?cia nos itens categorizados e m?tricas por categoria. Deixe itens sem r?tulo esperado fora do c?lculo.

M?tricas internas servem para selecionar candidatos ? inspe??o. A decis?o sobre a utilidade dos grupos e os nomes das categorias exige revis?o humana. A reserva s? ? um teste independente se os r?tulos e a taxonomia forem definidos sem consultar os resultados dessa avalia??o.

## Execu??o completa

Depois de revisar os par?metros, transfira os valores desejados para a se??o de par?metros do notebook 70 e execute aquele fluxo. Cada execu??o mant?m relat?rios, modelos e arquivos de revis?o em diret?rios pr?prios; use um novo identificador para um novo experimento.